# Sesión 9 — Segmentar municipios de Colombia: PCA + K-means + mapa (y una lectura honesta)

**Introducción al Análisis Avanzado de Datos con Python** · Univalle · Módulo 3 (cierre) · **se presenta E3**

## Objetivos de hoy
Al terminar la sesión usted podrá:
1. Encadenar el flujo no supervisado completo: **escalar → PCA para entender → K-means para agrupar → perfilar → validar** con algo que el algoritmo no vio (el mapa).
2. Elegir `k` combinando silueta, **estabilidad** y utilidad para la pregunta.
3. Describir cada grupo con su **perfil**, sin confundir el promedio del grupo con cada uno de sus miembros.
4. Decidir **qué variables** entran al clustering y explicar por qué meter un resumen junto con sus partes cambia los grupos.

## Dónde estamos en el pipeline

```
 ┌────────┐   ┌──────────────────┐   ┌────────┐   ┌────────────┐   ┌────────────────┐
 │ DATOS  │ → │ PREPROCESAMIENTO │ → │ MODELO │ → │ EVALUACIÓN │ → │ INTERPRETACIÓN │
 └────────┘   └──────────────────┘   └────────┘   └────────────┘   └────────────────┘
   ▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲
   HOY: el pipeline no supervisado completo, sobre datos de Colombia, de punta a punta.
```

🔍 **Predice** · ✏️ **Completa** (andamiaje **bajo**: usted escribe tramos completos) · 💥 **Trampa** · ⚡ **Mini-reto** · 📦 **Entrega E3**.

> **Datos:** Medida de Pobreza Multidimensional municipal del **Censo 2018** (DANE): para cada uno de los 1 122 municipios y áreas no municipalizadas, el % de personas en hogares con cada una de 15 **privaciones** (analfabetismo, sin acueducto, trabajo informal…) y el **IPM**, el índice que las resume. Coordenadas de la cabecera: DIVIPOLA (DANE, datos.gov.co).


## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

pd.set_option("display.max_columns", 40)
REPO = "https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/"
ARCHIVOS = {"municipios_ipm_2018.csv": "datos/municipios_ipm_2018.csv", "figuras.py": "tools/figuras.py"}

def asegurar_datos(archivo, url_repo):
    """Deja un archivo en el disco de Colab: lo baja del repositorio del curso si aún no está."""
    from pathlib import Path
    import requests
    if Path(archivo).exists():
        return True
    try:
        r = requests.get(url_repo, timeout=120)
        if r.ok:
            Path(archivo).write_bytes(r.content)
            return True
    except Exception:
        pass
    print(f"⚠️ No pude bajar {archivo}. Use el Plan B (celda siguiente).")
    return False

for archivo, ruta in ARCHIVOS.items():
    asegurar_datos(archivo, REPO + ruta)
import figuras as fg
print("Listo:", ", ".join(ARCHIVOS))

<details><summary><b>Plan B</b> — si la celda anterior avisó que no pudo bajar algún archivo</summary>

La celda siguiente revisa sola qué falta y solo en ese caso le pide subirlo.
</details>

In [ ]:
# Plan B: solo actúa si falta algún archivo. Con "Ejecutar todo" no se detiene.
from pathlib import Path
faltan = [a for a in ARCHIVOS if not Path(a).exists()]
if not faltan:
    print("✓ Todo disponible: no hace falta el Plan B.")
else:
    from google.colab import files
    print("⚠️ Faltan:", faltan, "→ súbalos")
    files.upload()

In [ ]:
fg.pipeline("todas", modulo=3, subtitulo="HOY: el pipeline no supervisado completo, de punta a punta");

## 1. Los datos: 15 formas de ser pobre

El **IPM** (índice de pobreza multidimensional) no mide ingresos: mide **privaciones**. Un hogar es pobre multidimensional si sufre al menos un tercio (ponderado) de estas 15:

| Dimensión | Privaciones (columna) |
|---|---|
| Educación | `analfabetismo`, `bajo_logro_educativo` |
| Niñez y juventud | `inasistencia_escolar`, `rezago_escolar`, `barreras_primera_infancia`, `trabajo_infantil` |
| Trabajo | `dependencia_economica`, `trabajo_informal` |
| Salud | `sin_aseguramiento_salud`, `barreras_salud` |
| Vivienda y servicios | `sin_agua_mejorada`, `sin_alcantarillado`, `pisos_inadecuados`, `paredes_inadecuadas`, `hacinamiento` |

Cada número es el **% de personas** del municipio que viven en hogares con esa privación.

In [ ]:
muni = pd.read_csv("municipios_ipm_2018.csv", dtype={"cod_mpio": str, "cod_dpto": str})
privaciones = list(muni.columns[10:])
print(muni.shape)
muni.head()

### 🔍 Predice
¿Qué departamento tiene el IPM promedio (entre sus municipios) **más alto**? ¿Y el más bajo, sin contar Bogotá?

**Tu predicción:** más alto ____ · más bajo ____

In [ ]:
muni.groupby("departamento")["ipm"].mean().sort_values().round(1).iloc[[0, 1, 2, -3, -2, -1]]

In [ ]:
fg.mapa_municipios(muni, valor="ipm", titulo="IPM por municipio (Censo 2018)");

**Ojo con un vacío:** `ipm_cabecera` tiene 20 vacíos. No es un error: son áreas no municipalizadas de la Amazonía y la Orinoquía que **no tienen cabecera urbana**. Es un vacío que informa (Sesión 2). Lo vamos a necesitar más adelante.

## 2. Entender antes de agrupar: PCA

### ✏️ Completa — escale y ajuste PCA (andamiaje bajo)
Escriba usted el código: escale las 15 privaciones con `StandardScaler` (guárdelas en `Z`), ajuste un `PCA()` con todos los componentes (guárdelo en `pca`) y dibuje la varianza explicada con `fg.pca_varianza`.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

Z = ...     # TODO: las 15 privaciones escaladas con StandardScaler
pca = ...   # TODO: PCA() ajustado con Z
fg.pca_varianza(pca, marcar=0.8, titulo="15 privaciones: ¿cuántas dimensiones de verdad?");

PC1 recoge **45 %** de la variación de 15 columnas: muchas privaciones van juntas (donde falta acueducto suele faltar alcantarillado, y hay más analfabetismo).

### 🔍 Predice
PC1 se construyó sin mirar el IPM. ¿Qué tan correlacionado cree que está con el IPM oficial del DANE (de −1 a 1)?

**Tu predicción:** ____

In [ ]:
scores = pca.transform(Z)
print("correlación PC1 – IPM:", round(np.corrcoef(scores[:, 0], muni["ipm"])[0, 1], 3))
fg.biplot(scores, pca.components_, privaciones, var_exp=pca.explained_variance_ratio_, top=7);

## 3. Agrupar: ¿cuántos grupos?

En la Sesión 7 vimos codo y silueta. Hoy agregamos una tercera pregunta: **¿los grupos sobreviven si cambio un poco los datos?** Para cada `k`, se re-entrena K-means 20 veces con el 80 % de los municipios y se mide cuánto se parecen los grupos a los originales (1 = idénticos).

In [ ]:
fg.estabilidad_k(Z, ks=range(2, 8));

### ✏️ Completa — K-means con k = 4
Entrene K-means sobre `Z` (`n_init=10`, `random_state=42`). Los números de cluster son nombres arbitrarios; para leer más fácil, la celda los **renumera** del menos al más pobre.

In [ ]:
from sklearn.cluster import KMeans

kmeans = ...   # TODO: K-means con k = 4 sobre Z
orden = muni["ipm"].groupby(kmeans.labels_).mean().sort_values().index      # del menos al más pobre
muni["grupo"] = pd.Series(kmeans.labels_).map({viejo: nuevo for nuevo, viejo in enumerate(orden)})
muni.groupby("grupo")["ipm"].agg(["count", "mean", "min", "max"]).round(1)

## 4. Leer los grupos: perfil y mapa

### 🔍 Predice
Antes de ver el perfil: ¿cree que los grupos 1 y 2 (IPM medio) se diferencian **solo** en el nivel de pobreza, o en el **tipo**? ¿En qué privación esperaría la mayor diferencia?

**Tu predicción:** ____

In [ ]:
fg.perfil_clusters(muni, muni["grupo"], ["ipm"] + privaciones);

In [ ]:
fg.mapa_municipios(muni, etiquetas=muni["grupo"], titulo="Cuatro tipos de municipio según sus privaciones");

In [ ]:
# Validación externa: la región NO se usó para agrupar
region = muni["cod_dpto"].map({"05": "Andina", "08": "Caribe", "11": "Andina", "13": "Caribe", "15": "Andina", "17": "Andina",
    "18": "Amazonía", "19": "Pacífica", "20": "Caribe", "23": "Caribe", "25": "Andina", "27": "Pacífica", "41": "Andina",
    "44": "Caribe", "47": "Caribe", "50": "Orinoquía", "52": "Pacífica", "54": "Andina", "63": "Andina", "66": "Andina",
    "68": "Andina", "70": "Caribe", "73": "Andina", "76": "Pacífica", "81": "Orinoquía", "85": "Orinoquía", "86": "Amazonía",
    "88": "Caribe", "91": "Amazonía", "94": "Amazonía", "95": "Amazonía", "97": "Amazonía", "99": "Orinoquía"})
pd.crosstab(region.rename("región"), muni["grupo"])

## 5. 💥 "Más columnas, más información"

Una idea muy natural: *"si agregamos el IPM y sus versiones urbana y rural, el clustering tiene más información y quedará mejor"*. Probemos.

In [ ]:
con_ipm = privaciones + ["ipm", "ipm_cabecera", "ipm_rural"]
try:
    KMeans(n_clusters=4, n_init=10, random_state=42).fit(StandardScaler().fit_transform(muni[con_ipm]))
except ValueError as e:
    print("❌", str(e).split(".")[0])

Primer tropiezo: K-means **no acepta vacíos**. Los 20 sin cabecera urbana rompen el algoritmo. Podríamos rellenarlos, pero ¿con qué? Un municipio sin cabecera no tiene "IPM urbano medio": no tiene ciudad. Dejamos `ipm_cabecera` por fuera y seguimos con `ipm` e `ipm_rural`.

### 🔍 Predice
Con esas 2 columnas más (17 en total), ¿cuántos de los 1 122 municipios cree que **cambian de grupo**?

**Tu predicción:** ____

In [ ]:
con_ipm = privaciones + ["ipm", "ipm_rural"]
km2 = KMeans(n_clusters=4, n_init=10, random_state=42).fit(StandardScaler().fit_transform(muni[con_ipm]))
orden2 = muni["ipm"].groupby(km2.labels_).mean().sort_values().index
grupo2 = pd.Series(km2.labels_).map({viejo: nuevo for nuevo, viejo in enumerate(orden2)})
print("municipios que cambian de grupo:", (muni["grupo"] != grupo2).sum())
pd.crosstab(muni["grupo"].rename("sin IPM"), grupo2.rename("con IPM"))

## 6. Lo que se puede y lo que NO se puede concluir

| ✓ Se puede decir | ✗ No se puede decir |
|---|---|
| Hay cuatro tipos de municipio estables según sus privaciones en 2018 | que esos cuatro grupos "existen" en la naturaleza (otro `k` u otras columnas darían otros) |
| El grupo 2 se distingue por vivienda y servicios, y es mayoritariamente caribeño | que ser caribeño **causa** pisos de tierra (el clustering no habla de causas) |
| Quibdó comparte perfil con municipios del Caribe rural | que **cada persona** de un municipio del grupo 3 es pobre (eso es la **falacia ecológica**: confundir el promedio del grupo con sus individuos) |
| Los grupos coinciden con regiones que el algoritmo no vio | que 286 municipios "ricos" son el 25 % de la población: cada municipio cuenta 1, sea Bogotá o Pana Pana |

Y un límite de los datos: **es 2018**. Un censo es una foto; el clustering describe esa foto.

## 7. 📦 Entrega E3 — análisis no supervisado de su dataset

Sobre **su** dataset (el de E1 y E2), en un notebook de Colab. Ahora **sin** usar la *y*: la deja guardada para leer los grupos al final.

| # | Sección | Qué debe contener |
|---|---|---|
| 1 | Columnas elegidas | Cuáles entran al clustering **y por qué** (¿hay resúmenes junto a sus partes? ¿identificadores? ¿la *y*?) |
| 2 | Preparación | Escalado (obligatorio para K-means/DBSCAN/PCA) · qué hizo con los vacíos y por qué |
| 3 | PCA | Varianza explicada · biplot · qué significa PC1 (y PC2 si aplica) |
| 4 | Clustering | K-means (o DBSCAN) · cómo eligió `k` (codo, silueta, **estabilidad**, utilidad) |
| 5 | Perfil | Tabla o `fg.perfil_clusters` · un nombre para cada grupo |
| 6 | Validación externa | Compare con algo que no usó: su *y*, una categoría, el tiempo, el mapa |
| 7 | **Lo que no se puede concluir** | Mínimo tres frases: causalidad, estabilidad, individuos vs grupos, dependencia de las columnas |

Rúbrica (1–4 cada uno): corrección técnica · decisiones justificadas (columnas, escala, `k`) · interpretación honesta · claridad.

Plantilla mínima (cambie lo que está en MAYÚSCULAS):

In [ ]:
# ---------- PLANTILLA E3 ----------
# df = pd.read_csv("/content/drive/MyDrive/SU_CARPETA/SUS_DATOS.csv")
# columnas = ["COL_1", "COL_2", "COL_3"]                # numéricas, sin la y ni identificadores
# Z = StandardScaler().fit_transform(df[columnas].dropna())
#
# pca = PCA().fit(Z);  fg.pca_varianza(pca)
# fg.biplot(pca.transform(Z), pca.components_, columnas, var_exp=pca.explained_variance_ratio_)
#
# fg.estabilidad_k(Z, ks=range(2, 8))
# km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(Z)
# fg.perfil_clusters(df.dropna(subset=columnas), km.labels_, columnas)
# pd.crosstab(df.dropna(subset=columnas)["SU_Y"], km.labels_)    # validación externa

## ⚡ Mini-reto (opcional)

1. Repita el clustering usando **solo las 5 privaciones de vivienda y servicios**. ¿Cómo cambia el mapa? ¿Qué pregunta respondería ese clustering?
2. Pruebe DBSCAN sobre las dos primeras componentes de PCA (`scores[:, :2]`). ¿Qué municipios marca como **ruido**? ¿Son los mismos del grupo 3?

In [ ]:
# Su análisis aquí

## 8. Cierre del Módulo 3

![El flujo no supervisado](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S09/img/flujo_no_supervisado.png)

**Las tres trampas del módulo, juntas:**

| Sesión | Decisión del analista | Qué pasó |
|---|---|---|
| S7 | unidades de medida | K-means agrupó pingüinos por gramos |
| S8 | el dibujo (perplexity, semilla) | t-SNE inventó grupos en ruido puro |
| S9 | qué columnas entran | el IPM votó dos veces y movió 75 municipios |

En el aprendizaje no supervisado **no hay una *y* que delate el error**. Por eso cada decisión se justifica y cada conclusión se valida con algo externo.

**Para la Sesión 10 — Módulo 4:** volvemos a los modelos con *y*. ¿Cómo saber si un modelo es bueno **de verdad** y no solo con la partición que nos tocó? **Validación cruzada.**